<a href="https://colab.research.google.com/github/yerim410/AI/blob/main/%EA%B3%BC%EC%A0%9C4_202412330_%EC%9D%B4%EC%98%88%EB%A6%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 과제 4 — RL 미니환경: 리턴 · GAE · PPO 클립 손실 · 레벨 표 · BC→RL 미세조정 · 보상 설계
**AI Application · 스마트모빌리티학과 · 배포 2026-11-04 · 마감 11/10(화) 23:59**

10주차 노트북의 PPO 를 부품 단위로 직접 구현하고, 사전학습 정책과 여러분의 정책을 챌린지 레벨에서 비교합니다.

| 문제 | 내용 | 배점 |
|---|---|---|
| 1 | 할인 리턴 G_t 계산 | 15 |
| 2 | GAE 구현 | 15 |
| 3 | PPO 클립 손실 | 15 |
| 4 | 사전학습 정책 × 레벨 표 | 15 |
| 5 | PPO 업데이트 빈칸 5개 + 짧은 미세조정 실행 | 15 |
| 6 | 보상 함수 재설계 + 성질 검사 | 15 |
| — | 런타임 → 모두 실행 오류 없음 | 10 |
| 심화 | bc_coef ∈ {0, 0.3, 1.0} 비교 + 3줄 해석 | +10 |

**제출물**: 이 노트북 + `rl_log.csv` + `rl_result.json` (+ 선택 `my_policy_rl.onnx`). 채점 셀 수정 시 0점. CPU 5~7분.

In [25]:
# ============ 0. 준비 (재접속 시 여기부터) ============
import importlib.util, subprocess, sys, os, time, json, copy
for pkg in ["onnx", "onnxruntime", "pandas"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
try:
    import tinytrack as tt
except ImportError:
    from google.colab import files
    print("tinytrack.py 를 업로드하세요"); files.upload(); import tinytrack as tt
assert tt.__version__ >= "0.3.0", "tinytrack 0.3.0 이상 필요"
try:
    from google.colab import drive
    drive.mount("/content/drive"); ROOT = Path("/content/drive/MyDrive/AIApp")
except Exception:
    ROOT = Path("./outputs")
SAVE_DIR = ROOT / "week10"; SAVE_DIR.mkdir(parents=True, exist_ok=True)
STUDENT = "202412330_이예림"     # ← 본인 것으로
torch.manual_seed(0); np.random.seed(0)
MODELS = Path("models") if Path("models/ppo_bc12.onnx").exists() else Path(".")
for f in ("ppo_scratch.onnx", "ppo_bc12.onnx", "base_pilotnet_dr.onnx", "base_pilotnet.pt"):
    if not (MODELS / f).exists():
        from google.colab import files
        print(f"{f} 업로드 (e2e_starter/models/)"); files.upload()
for cand in [ROOT / "week10" / "start_policy.pt", ROOT / "week07" / "my_policy_v2.pt", ROOT / "week06" / "my_policy_v1.pt"]:
    if cand.exists():
        start = tt.load_model(cand); print("시작 정책:", cand.name); break
else:
    D = tt.rebalance(tt.merge(*[tt.collect(tt.ExpertPolicy(), tt.Track.preset(n), 500, seed=10 * ord(n) + k, noise_inject=ni)
                                for n in "ABC" for k, ni in enumerate([0.0, 0.4, 0.8])]))
    tr, va = tt.split_by_run(D, 0.2, seed=0); start = tt.load_model(MODELS / "base_pilotnet.pt")
    tt.train_bc(start, tr, va, epochs=3, lr=3e-4, augment=tt.make_augment(seed=0), verbose=False); print("시작 정책: 즉석 미세조정")
PUBLIC = ["A", "B", "C"]; HARD = dict(every=50, d=0.42, psi=0.36)
FAST = os.environ.get("NB_FAST") == "1"
print("tinytrack", tt.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
시작 정책: start_policy.pt
tinytrack 0.3.1


## 문제 1 — 할인 리턴 (15점)
`discounted_returns(rewards, gamma)` : 보상 배열 r_0..r_{T−1} 에 대해 G_t = Σ_{k≥0} γ^k r_{t+k} 를 반환하세요 (길이 T). **뒤에서 앞으로** 누적하면 O(T) 입니다.

In [26]:
def discounted_returns(rewards, gamma=0.99):
    G = np.zeros(len(rewards), dtype=np.float64); running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running                      # r_t + γ · (다음 스텝의 리턴)
        G[t] = running
    return G

print(discounted_returns(np.array([1.0, 1.0, 1.0, -5.0]), 0.9))

[-0.935 -2.15  -3.5   -5.   ]


In [27]:
# ---- 채점 1 (수정 금지) ----
_r = np.random.RandomState(1).randn(50)
_ref = np.array([sum(0.97 ** k * _r[t + k] for k in range(50 - t)) for t in range(50)])
assert np.allclose(discounted_returns(_r, 0.97), _ref, atol=1e-6), "리턴 값이 참조와 다릅니다"
assert np.allclose(discounted_returns(_r, 0.0), _r), "γ=0 이면 리턴 = 보상"
print("문제 1 통과!")

문제 1 통과!


## 문제 2 — GAE (15점)
`gae(rewards, values, dones, next_value, gamma, lam)` : δ_t = r_t + γ·V_{t+1}·(1−done_{t+1}) − V_t,  A_t = δ_t + γλ·(1−done_{t+1})·A_{t+1}.
`dones[t]` 는 스텝 t 가 시작될 때 '직전 에피소드가 끝났는가' (10주차 노트북과 같은 규약), 마지막 스텝 다음의 가치는 `next_value`, 다음 done 은 0 으로 둡니다.

In [28]:
def gae(rewards, values, dones, next_value, gamma=0.99, lam=0.95):
    T = len(rewards); adv = np.zeros(T); last = 0.0
    for t in reversed(range(T)):
        nonterm = 1.0 - (0.0 if t == T - 1 else dones[t + 1])
        nv = next_value if t == T - 1 else values[t + 1]
        delta = rewards[t] + gamma * nv * nonterm - values[t]                                        # TD 오차
        adv[t] = last = delta + gamma * lam * nonterm * last                                # δ + γλ·nonterm·이전 누적
    return adv

r_, v_, d_ = np.array([1., 1., 1., 1.]), np.array([0.5, 0.6, 0.7, 0.8]), np.array([0., 0., 1., 0.])
print(gae(r_, v_, d_, next_value=0.9))

[1.4702    0.4       2.1180855 1.091    ]


In [29]:
# ---- 채점 2 (수정 금지) ----
_rs = np.random.RandomState(2); _r, _v = _rs.randn(30), _rs.randn(30); _d = np.zeros(30); _d[10] = 1.0
_a0 = gae(_r, _v, _d, 0.3, 0.99, 0.0)                                   # λ=0 → 1스텝 TD
_nv = np.append(_v[1:], 0.3); _nt = 1.0 - np.append(_d[1:], 0.0)
assert np.allclose(_a0, _r + 0.99 * _nv * _nt - _v, atol=1e-8), "λ=0 이면 1스텝 TD 오차와 같아야 합니다"
_a1 = gae(_r, _v, np.zeros(30), 0.0, 1.0, 1.0)                          # γ=λ=1, done 없음 → 리턴 − V
assert np.allclose(_a1, np.cumsum(_r[::-1])[::-1] - _v, atol=1e-8), "γ=λ=1 이면 (리턴 − V) 여야 합니다"
_a = gae(_r, _v, _d, 0.3)
assert abs(_a[9] - (_r[9] - _v[9])) < 1e-8, "done 다음 스텝으로 부트스트랩이 새고 있습니다 (nonterm 확인)"
print("문제 2 통과!")

문제 2 통과!


## 문제 3 — PPO 클립 손실 (15점)
`ppo_clip_loss(ratio, adv, clip=0.2)` : `mean( max( −adv·ratio, −adv·clip(ratio, 1−ε, 1+ε) ) )` 를 torch 로 구현하세요 (ratio 는 미분 가능 텐서).

In [30]:
def ppo_clip_loss(ratio, adv, clip=0.2):
    unclipped = -ratio * adv
    clipped = -torch.clamp(ratio, 1.0 - clip, 1.0 + clip) * adv
    return torch.max(unclipped, clipped).mean()

ratio = torch.tensor([0.5, 1.0, 1.5], requires_grad=True); adv = torch.tensor([1.0, 1.0, 1.0])
print(ppo_clip_loss(ratio, adv))

tensor(-0.9000, grad_fn=<MeanBackward0>)


In [31]:
# ---- 채점 3 (수정 금지) ----
_ratio = torch.tensor([0.5, 0.9, 1.0, 1.1, 1.5], requires_grad=True); _adv = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0])
_l = ppo_clip_loss(_ratio, _adv); _l.backward()
assert abs(_l.item() - (-(0.5 + 0.9 + 1.0 + 1.1 + 1.2) / 5)) < 1e-6, "손실 값이 참조와 다릅니다 (max 는 비관적 쪽: ratio 0.5 는 클립 안 됨)"
assert abs(_ratio.grad[4].item()) < 1e-8 and abs(_ratio.grad[2].item() + 0.2) < 1e-6, "A>0 에서 ratio>1+ε 이면 기울기 0, 안이면 −A/N 이어야 합니다"
_ratio2 = torch.tensor([0.5, 1.5], requires_grad=True); ppo_clip_loss(_ratio2, torch.tensor([-1.0, -1.0])).backward()
assert abs(_ratio2.grad[0].item()) < 1e-8 and _ratio2.grad[1].item() > 0, "A<0 에서 ratio<1−ε 이면 기울기 0 이어야 합니다"
print("문제 3 통과!")

문제 3 통과!


## 문제 4 — 사전학습 정책 × 레벨 표 (15점)
`level_table(policies, levels)` : `{이름: 정책}` 과 레벨 목록을 받아 각 정책을 공개 트랙 A/B/C 에서 `tt.evaluate(pol, PUBLIC, seed=0, **tt.level_eval_kwargs(L, style_seed=7))` 로 평가한 DataFrame(index=정책, columns=레벨)을 반환하세요. 정책은 `ppo_scratch`, `ppo_bc12`, `start`(시작 정책) 세 개, 레벨은 L1·L2·L4.

In [32]:
def level_table(policies, levels):
    rows = []
    for name, pol in policies.items():
        row = {"policy": name}
        for L in levels:
            row[L] = round(tt.evaluate(pol, PUBLIC, seed=0, **tt.level_eval_kwargs(L, style_seed=7))["score"], 1)          # 레벨 조건으로 평가
        rows.append(row)
    return pd.DataFrame(rows).set_index("policy")

policies = {"start": tt.TorchPolicy(start), "ppo_scratch": tt.OnnxPolicy(MODELS / "ppo_scratch.onnx"), "ppo_bc12": tt.OnnxPolicy(MODELS / "ppo_bc12.onnx")}
t0 = time.time(); lv = level_table(policies, ["L1", "L2", "L4"]); print(lv.to_string(), f"({time.time()-t0:.0f}s)")

               L1    L2    L4
policy                       
start        86.7  17.3  57.8
ppo_scratch  90.5   9.6  44.4
ppo_bc12     88.3  23.3  72.0 (121s)


In [33]:
# ---- 채점 4 (수정 금지) ----
assert lv.shape == (3, 3) and list(lv.columns) == ["L1", "L2", "L4"], "3×3 표 (행 정책, 열 레벨)"
assert lv.loc["ppo_scratch", "L1"] >= 80, "ppo_scratch 는 L1 에서 80 이상이어야 합니다 (평가 조건 확인)"
assert lv.loc["ppo_bc12", "L4"] > lv.loc["ppo_scratch", "L4"] + 10, "L4(속도 1.3×) 에서 ppo_bc12 가 ppo_scratch 보다 10점 이상 높아야 합니다 (level_eval_kwargs 사용 여부)"
print("문제 4 통과!")

문제 4 통과!


## 문제 5 — PPO 업데이트 빈칸 5개 + 짧은 미세조정 (15점)
아래 제공 코드(벡터 환경·Agent·롤아웃·GAE)는 10주차 노트북과 같습니다. **업데이트 부분의 빈칸 5개**를 채우고 8,000 스텝(FAST 4,000)을 돌려 로그를 남기세요.

In [34]:
# ---- 제공 코드 (수정 불필요) ----
def to_tensor(obs_batch):
    return torch.from_numpy(np.concatenate([tt.preprocess(o) for o in obs_batch]))

class VecTinyTrack:
    def __init__(self, n, tracks, disturb, seed=0, reset_spread=0.15):
        self.rng = np.random.RandomState(seed); self.spread = reset_spread
        self.envs = [tt.Env(tt.Track.preset(tracks[i % len(tracks)]), disturb=disturb, seed=seed + i, max_steps=1500) for i in range(n)]
        self.ep_ret = np.zeros(n); self.finished = []
    def _reset(self, i):
        return self.envs[i].reset(d0=float(np.clip(self.rng.normal(0, self.spread), -0.4, 0.4)), psi0=float(np.clip(self.rng.normal(0, self.spread), -0.5, 0.5)))
    def reset(self): return np.stack([self._reset(i) for i in range(len(self.envs))])
    def step(self, actions):
        obs, rews, dones = [], [], []
        for i, (e, a) in enumerate(zip(self.envs, actions)):
            o, r, term, trunc, info = e.step(float(a)); self.ep_ret[i] += r
            if term or trunc:
                self.finished.append((self.ep_ret[i], info["progress"] / e.track.length)); self.ep_ret[i] = 0; o = self._reset(i)
            obs.append(o); rews.append(r); dones.append(float(term))
        return np.stack(obs), np.array(rews, np.float32), np.array(dones, np.float32)

class Agent(nn.Module):
    def __init__(self, start_model, init_log_std=-1.5):
        super().__init__(); self.actor = copy.deepcopy(start_model)
        c = tt.make_model(); self.critic = nn.Sequential(c.features, nn.Flatten(), nn.Linear(64 * 18, 100), nn.ReLU(), nn.Linear(100, 1))
        self.log_std = nn.Parameter(torch.full((1,), float(init_log_std)))
    def dist(self, x):
        mean = self.actor(x).squeeze(-1); return torch.distributions.Normal(mean, self.log_std.exp().expand_as(mean)), mean
    def value(self, x): return self.critic(x).squeeze(-1)

class ActorPolicy:
    def __init__(self, actor): self.actor = actor
    def __call__(self, obs, state=None):
        with torch.no_grad(): return float(np.clip(self.actor(torch.from_numpy(tt.preprocess(obs))).item(), -1, 1))

def rollout_and_gae(agent, venv, next_obs, next_done, T, gamma=0.99, lam=0.95, reward_scale=0.1):
    n = len(venv.envs)
    obs_buf = torch.zeros((T, n, 3, tt.H, tt.W)); act_buf = torch.zeros((T, n)); logp_buf = torch.zeros((T, n))
    rew_buf = torch.zeros((T, n)); done_buf = torch.zeros((T, n)); val_buf = torch.zeros((T, n))
    agent.eval()
    for t in range(T):
        with torch.no_grad():
            dist_, _ = agent.dist(next_obs); a = dist_.sample(); v = agent.value(next_obs)
        obs_buf[t], act_buf[t], logp_buf[t], val_buf[t], done_buf[t] = next_obs, a, dist_.log_prob(a), v, next_done
        o, r, dn = venv.step(a.numpy()); rew_buf[t] = torch.from_numpy(r) * reward_scale; next_obs = to_tensor(o); next_done = torch.from_numpy(dn)
    with torch.no_grad():
        next_v = agent.value(next_obs); adv = torch.zeros_like(rew_buf); last = torch.zeros(n)
        for t in reversed(range(T)):
            nonterm = 1.0 - (next_done if t == T - 1 else done_buf[t + 1]); nv = next_v if t == T - 1 else val_buf[t + 1]
            delta = rew_buf[t] + gamma * nv * nonterm - val_buf[t]; adv[t] = last = delta + gamma * lam * nonterm * last
    return (obs_buf, act_buf, logp_buf, adv, adv + val_buf, val_buf), next_obs, next_done
print("제공 코드 준비")

제공 코드 준비


In [35]:
def ppo_update(agent, prior, opt, buffers, it, clip=0.2, epochs=2, minibatches=2, critic_warmup=5, bc_coef=0.3):
    obs_buf, act_buf, logp_buf, adv, ret, val_buf = buffers
    b_obs, b_act, b_logp, b_adv, b_ret, b_val = (x.reshape(-1, *x.shape[2:]) for x in buffers)
    batch = len(b_act); idx = np.arange(batch); mb = batch // minibatches; clipfracs = []
    agent.train()
    for _ in range(epochs):
        np.random.shuffle(idx)
        for s in range(0, batch, mb):
            m = idx[s:s + mb]
            dist_, mean = agent.dist(b_obs[m]); newlogp = dist_.log_prob(b_act[m]); newv = agent.value(b_obs[m])
            ratio = (newlogp - b_logp[m]).exp()                                                        # ① 새/옛 로그확률 차이의 exp
            madv = (b_adv[m] - b_adv[m].mean()) / (b_adv[m].std() + 1e-8)
            pg = ppo_clip_loss(ratio, madv, clip)                                                           # ② 클립 손실 (문제 3 의 함수 사용 가능)
            v_clipped = b_val[m] + torch.clamp(newv - b_val[m], -clip, clip)
            vl = 0.5 * torch.max((newv - b_ret[m]) ** 2, (v_clipped - b_ret[m]) ** 2).mean()
            if it <= critic_warmup:
                loss = 0.5 * vl                                                     # ③ 워밍업: 가치 손실만 (0.5 배)
            else:
                with torch.no_grad(): pm = prior(b_obs[m]).squeeze(-1)
                loss = pg + 0.5 * vl + bc_coef * ((mean - pm) ** 2).mean()                                                     # ④ pg + 0.5·vl + bc_coef·MSE(mean, pm)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(agent.parameters(), 0.5); opt.step()   # ⑤ 파라미터 갱신
            clipfracs.append(((ratio - 1).abs() > clip).float().mean().item())
    return float(np.mean(clipfracs))

In [36]:
# ---- 실행 (수정 불필요) : 8 환경 × 64 스텝, 8,000 스텝 (FAST 4,000) ----
TOTAL = 4000 if FAST else 8000; N_ENVS, T = 8, 64
torch.manual_seed(0); np.random.seed(0)
venv = VecTinyTrack(N_ENVS, PUBLIC, HARD, seed=0); agent = Agent(start); prior = copy.deepcopy(agent.actor).eval()
for p in prior.parameters(): p.requires_grad_(False)
opt = torch.optim.Adam([dict(params=list(agent.actor.parameters()) + [agent.log_std], lr=1e-4), dict(params=agent.critic.parameters(), lr=3e-4)], eps=1e-5)
next_obs, next_done = to_tensor(venv.reset()), torch.zeros(N_ENVS)
start_hard = tt.evaluate(tt.TorchPolicy(start), ["B"], seed=0, disturb=HARD)["score"]
log = []; t0 = time.time(); best = (start_hard, copy.deepcopy(agent.actor))
for it in range(1, TOTAL // (N_ENVS * T) + 1):
    buffers, next_obs, next_done = rollout_and_gae(agent, venv, next_obs, next_done, T)
    cf = ppo_update(agent, prior, opt, buffers, it, critic_warmup=3)
    row = dict(iter=it, steps=it * N_ENVS * T, std=round(agent.log_std.exp().item(), 3), clipfrac=round(cf, 3), sec=round(time.time() - t0))
    if it % 4 == 0 or it == TOTAL // (N_ENVS * T):
        agent.eval(); row["score_hard"] = round(tt.evaluate(ActorPolicy(agent.actor), ["B"], seed=0, disturb=HARD)["score"], 1)
        if row["score_hard"] > best[0]: best = (row["score_hard"], copy.deepcopy(agent.actor))
    log.append(row); print(row)
pd.DataFrame(log).to_csv(SAVE_DIR / "rl_log.csv", index=False)
print(f"시작 {start_hard:.1f} → 최고 {best[0]:.1f} (트랙 B, 1.2×)  {time.time()-t0:.0f}s")

{'iter': 1, 'steps': 512, 'std': 0.223, 'clipfrac': 0.0, 'sec': 11}
{'iter': 2, 'steps': 1024, 'std': 0.223, 'clipfrac': 0.0, 'sec': 22}
{'iter': 3, 'steps': 1536, 'std': 0.223, 'clipfrac': 0.0, 'sec': 33}
{'iter': 4, 'steps': 2048, 'std': 0.223, 'clipfrac': 0.182, 'sec': 49, 'score_hard': 41.9}
{'iter': 5, 'steps': 2560, 'std': 0.223, 'clipfrac': 0.058, 'sec': 71}
{'iter': 6, 'steps': 3072, 'std': 0.223, 'clipfrac': 0.013, 'sec': 88}
{'iter': 7, 'steps': 3584, 'std': 0.223, 'clipfrac': 0.008, 'sec': 105}
{'iter': 8, 'steps': 4096, 'std': 0.223, 'clipfrac': 0.009, 'sec': 122, 'score_hard': 7.8}
{'iter': 9, 'steps': 4608, 'std': 0.223, 'clipfrac': 0.036, 'sec': 139}
{'iter': 10, 'steps': 5120, 'std': 0.223, 'clipfrac': 0.007, 'sec': 155}
{'iter': 11, 'steps': 5632, 'std': 0.223, 'clipfrac': 0.053, 'sec': 171}
{'iter': 12, 'steps': 6144, 'std': 0.223, 'clipfrac': 0.006, 'sec': 188, 'score_hard': 43.1}
{'iter': 13, 'steps': 6656, 'std': 0.223, 'clipfrac': 0.0, 'sec': 208}
{'iter': 14, 'st

In [37]:
# ---- 채점 5 (수정 금지) ----
assert (SAVE_DIR / "rl_log.csv").exists() and len(log) >= 5, "로그 5행 이상"
assert all(np.isfinite([r["std"] for r in log])) and 0.05 < log[-1]["std"] < 0.6, "σ 가 비정상입니다 (발산 또는 소멸)"
assert all(0 <= r["clipfrac"] <= 1 for r in log) and np.mean([r["clipfrac"] for r in log if r["iter"] > 3]) < 0.6, "clipfrac 이 너무 큽니다 — ratio 계산 확인"
assert best[0] >= start_hard - 10, "학습 후 점수가 시작보다 10점 이상 떨어졌습니다 (loss 조합·워밍업·BC 항 확인)"
print("문제 5 통과!")

문제 5 통과!


## 문제 6 — 보상 함수 재설계 + 성질 검사 (15점)
`my_reward(d, psi, steer, offtrack)` 을 설계하세요. 채점 셀이 검사하는 성질: ① |d| 에 대해 단조 감소 ② 이탈이면 1 이상 더 낮음 ③ 같은 상태에서 |steer| 가 크면 낮거나 같음 ④ 스텝 보상 ≤ 1 ⑤ 중앙·직진·비이탈에서 최대. (예: 1 − 2|d| − 0.1|steer| − 5·이탈 — 다른 설계도 성질만 만족하면 됩니다)

In [38]:
def my_reward(d, psi, steer, offtrack):
    r = 1.0 - 2.0 * abs(d) - 0.1 * abs(steer) - 5.0 * offtrack
    return float(r)

print(my_reward(0.0, 0.0, 0.0, False), my_reward(0.3, 0.0, 0.5, False), my_reward(0.5, 0.0, 0.0, True))

1.0 0.35000000000000003 -5.0


In [39]:
# ---- 채점 6 (수정 금지) ----
_vals = [my_reward(x, 0.0, 0.0, False) for x in (0.0, 0.1, 0.2, 0.3, 0.4)]
assert all(_vals[i] > _vals[i + 1] for i in range(4)), "① |d| 가 커질수록 보상이 줄어야 합니다"
assert my_reward(0.2, 0.0, 0.0, True) <= my_reward(0.2, 0.0, 0.0, False) - 1.0, "② 이탈 페널티가 1 이상이어야 합니다"
assert my_reward(0.1, 0.0, 0.8, False) <= my_reward(0.1, 0.0, 0.0, False), "③ 조향이 크면 보상이 낮거나 같아야 합니다"
assert my_reward(0.0, 0.0, 0.0, False) <= 1.0 + 1e-9, "④ 스텝 보상은 1 을 넘지 않게"
assert my_reward(0.0, 0.0, 0.0, False) >= max(my_reward(0.3, 0.2, 0.5, False), my_reward(0.0, 0.0, 0.0, True)), "⑤ 중앙·직진·비이탈이 최대"
print("문제 6 통과!")

문제 6 통과!


## 심화 (선택, +10) — bc_coef ∈ {0, 0.3, 1.0}
같은 4,000 스텝을 세 가지 bc_coef 로 돌려 (1.0× score, 1.2× score) 를 표로 만들고 3줄로 해석하세요. bc_coef=0 은 망각, 1.0 은 못 배움이 나타나는지.

In [40]:
rows = []
for bc in (0.0, 0.3, 1.0):
    torch.manual_seed(0); np.random.seed(0)
    venv_ = VecTinyTrack(N_ENVS, PUBLIC, HARD, seed=0); ag = Agent(start); pr = copy.deepcopy(ag.actor).eval()
    for p in pr.parameters(): p.requires_grad_(False)
    op = torch.optim.Adam([dict(params=list(ag.actor.parameters()) + [ag.log_std], lr=1e-4), dict(params=ag.critic.parameters(), lr=3e-4)], eps=1e-5)
    no, nd = to_tensor(venv_.reset()), torch.zeros(N_ENVS)
    for it in range(1, 4000 // (N_ENVS * T) + 1):
        bufs, no, nd = rollout_and_gae(ag, venv_, no, nd, T); ppo_update(ag, pr, op, bufs, it, critic_warmup=3, bc_coef=bc)
    ag.eval(); pol = ActorPolicy(ag.actor)
    rows.append(dict(bc_coef=bc, normal=round(tt.evaluate(pol, PUBLIC, seed=0)["score"], 1), hard=round(tt.evaluate(pol, PUBLIC, seed=0, disturb=HARD)["score"], 1)))
    print(rows[-1])
print(pd.DataFrame(rows).to_string(index=False))

{'bc_coef': 0.0, 'normal': 87.8, 'hard': 32.0}
{'bc_coef': 0.3, 'normal': 87.9, 'hard': 34.6}
{'bc_coef': 1.0, 'normal': 87.2, 'hard': 45.6}
 bc_coef  normal  hard
     0.0    87.8  32.0
     0.3    87.9  34.6
     1.0    87.2  45.6


```markdown
**심화 해석 (3줄):**
1. bc_coef=0.0 (모방 없음): 기존 지식을 잃어버리는 망각이 발생해 hard 환경에서 점수가 가장 낮음.
2. bc_coef=1.0 (강한 모방): 안전하게 기존 능력을 보존하므로 어려운 상황을 가장 우수하게 돌파함.
3. 짧은 학습 상황에서는 사전학습 지식을 강하게 묶어둘수록 챌린지 주행에 유리함.
```

## 결과 저장 · 제출

In [41]:
onnx_path = tt.export_onnx(best[1].cpu().eval(), SAVE_DIR / "my_policy_rl.onnx")
result = dict(student=STUDENT, level_table=lv.to_dict(), start_hard=start_hard, best_hard=best[0], steps=int(log[-1]["steps"]), onnx=str(onnx_path))
json.dump(result, open(SAVE_DIR / "rl_result.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("저장:", SAVE_DIR / "rl_result.json", "| 제출물: 이 노트북, rl_log.csv, rl_result.json (+ my_policy_rl.onnx)")

저장: /content/drive/MyDrive/AIApp/week10/rl_result.json | 제출물: 이 노트북, rl_log.csv, rl_result.json (+ my_policy_rl.onnx)


## 제출 방법
1. **런타임 → 모두 실행** 오류 없음 + 채점 셀 6개 `통과!`
2. `과제4_학번_이름.ipynb` 로 저장
3. ① GitHub 개인 저장소 push ② e-Campus ③ 이메일 `[AI응용] 과제4_학번_이름` (저장소 링크). 첨부 제출도 인정
4. **리더보드**: `my_policy_rl.onnx` 를 제출하면 1.0× / 1.2× 두 열로 게시

**마감 11/10(화) 23:59.**